# Small Reasoning LLM Lab — Model Evaluation

**How to use this notebook:**
1. Go to `Runtime → Change runtime type → T4 GPU`
2. Click `Runtime → Run all`
3. Wait — everything is automatic

You do **not** need to edit any code or find any file paths.

If the trained checkpoint (`best.pt`) is not found, the notebook will
automatically search Google Drive and offer to train if needed.

---
**What this notebook measures:**
- Test accuracy on 500 unseen arithmetic problems
- Generalization across 5 difficulty levels
- Checkpoint progression (init → early → mid → final → best)
- Actual model outputs including failures
- Interactive inference so you can type your own problems

---
## Step 1 — Set up the repository and install dependencies

In [ ]:
# ── Clone / update the repository and install dependencies ────────────────
# This cell is fully automatic. Do not edit.

import os, sys, subprocess

REPO_URL  = 'https://github.com/sinor77/small-llm-lab.git'
REPO_NAME = 'small-llm-lab'

def _git(*args, cwd=None):
    r = subprocess.run(['git'] + list(args), cwd=cwd or os.getcwd(),
                       capture_output=True, text=True)
    if r.returncode == 0:
        for line in r.stdout.strip().splitlines(): print(f'  git: {line}')
    else:
        print(f'  git: {r.stderr.strip()[:300]}')

if os.path.exists('model') and os.path.exists('training'):
    print('Already in repo directory. Pulling latest code...')
    _git('pull', 'origin', 'main')
    REPO_ROOT = os.getcwd()
elif os.path.exists(REPO_NAME):
    print('Repo found. Pulling latest code...')
    _git('pull', 'origin', 'main', cwd=REPO_NAME)
    os.chdir(REPO_NAME)
    REPO_ROOT = os.getcwd()
else:
    print(f'Cloning {REPO_URL} ...')
    _git('clone', REPO_URL)
    os.chdir(REPO_NAME)
    REPO_ROOT = os.getcwd()

sys.path.insert(0, REPO_ROOT)

# Install tokenizers if missing
try:
    import tokenizers
    print(f'tokenizers {tokenizers.__version__} already installed.')
except ImportError:
    print('Installing tokenizers...')
    subprocess.run([sys.executable, '-m', 'pip', 'install',
                    'tokenizers>=0.15.0', '-q'], check=True)
    print('Done.')

print(f'\nRepository root: {REPO_ROOT}')
_git('log', '--oneline', '-3')

---
## Step 2 — Detect GPU and find checkpoint

In [ ]:
# ── Device detection ─────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    DEVICE = 'cuda'
    print(f'GPU:    {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    DEVICE = 'cpu'
    print('No GPU detected. Using CPU (evaluation will be slower).')
    print('For GPU: Runtime -> Change runtime type -> T4 GPU')

print(f'\nDevice: {DEVICE}')

In [ ]:
# ── Automatic checkpoint search ───────────────────────────────────────────────
# Searches: experiments/results/ -> /content -> Google Drive
# Do NOT edit this cell.

import glob, os, time

BEST_CK   = None
EXP_DIR   = None
CK_DIR    = None

def _find_checkpoints(pattern):
    hits = glob.glob(pattern, recursive=True)
    hits.sort(key=os.path.getmtime, reverse=True)
    return hits

def _set_paths(ck_path):
    global BEST_CK, CK_DIR, EXP_DIR
    BEST_CK = os.path.abspath(ck_path)
    CK_DIR  = os.path.dirname(BEST_CK)
    # Walk up to find the tokenizer/ sibling (= experiment root)
    candidate = CK_DIR
    for _ in range(5):
        if os.path.isdir(os.path.join(candidate, 'tokenizer')):
            EXP_DIR = candidate
            return
        candidate = os.path.dirname(candidate)
    EXP_DIR = os.path.dirname(CK_DIR)   # fallback

# 1. experiments/results/
hits = _find_checkpoints(
    os.path.join(REPO_ROOT, 'experiments', 'results', '*', 'checkpoints', 'best.pt')
)
if hits:
    _set_paths(hits[0])
    if len(hits) > 1:
        print(f'Found {len(hits)} experiment(s). Using newest:')
        for h in hits: print(f'  {h}')

# 2. Anywhere in /content
if not BEST_CK:
    print('Not found in experiments/. Searching /content ...')
    hits = _find_checkpoints('/content/**/best.pt')
    if hits:
        _set_paths(hits[0])

# 3. Google Drive
if not BEST_CK:
    try:
        from google.colab import drive
        print('Mounting Google Drive to search for best.pt ...')
        drive.mount('/content/drive', force_remount=False)
        hits = _find_checkpoints('/content/drive/MyDrive/**/best.pt')
        if hits:
            _set_paths(hits[0])
            print(f'Found on Drive: {BEST_CK}')
    except Exception as e:
        print(f'Drive search skipped: {e}')

if BEST_CK:
    print()
    print('=' * 55)
    print(f'Using checkpoint: {BEST_CK}')
    print(f'Experiment dir:   {EXP_DIR}')
    print('=' * 55)
else:
    print()
    print('=' * 55)
    print('  CHECKPOINT NOT FOUND')
    print('  best.pt was not found anywhere in this Colab session.')
    print('  Scroll to the next cell to train the model or restore it.')
    print('=' * 55)

In [ ]:
# ── Recovery: train if no checkpoint found ───────────────────────────────────
# If best.pt was found above, this cell does nothing.
# If not found, it runs training automatically (~10-30 min on T4).

if BEST_CK is None:
    print('No checkpoint found. Running training to create one...')
    print('This will take approximately 10-30 minutes on a T4 GPU.')
    print()

    from training.train import train
    from training.config import get_train_config_by_name

    train_config = get_train_config_by_name('colab_small')
    train(train_config)

    # Locate the newly created checkpoint
    new_best = os.path.join(REPO_ROOT, train_config.checkpoint_dir, 'best.pt')
    if os.path.exists(new_best):
        _set_paths(new_best)
        print(f'Training complete. Checkpoint: {BEST_CK}')
    else:
        # Last resort: search again
        hits = _find_checkpoints(
            os.path.join(REPO_ROOT, 'experiments', 'results', '*', 'checkpoints', 'best.pt')
        )
        if hits:
            _set_paths(hits[0])
            print(f'Training complete. Checkpoint: {BEST_CK}')
        else:
            raise RuntimeError(
                'Training finished but best.pt was not found. '
                'Check training logs above for errors.'
            )
else:
    print(f'Checkpoint already found. Skipping training.\n{BEST_CK}')

---
## Step 3 — Load the model

In [ ]:
# ── Load model, tokenizer, metadata ──────────────────────────────────────────
import time, json
from inference.generate import load_model_and_tokenizer, generate_answer, solve_problem
from data.generators.arithmetic import ArithmeticGenerator
from evaluation.benchmark import run_benchmark, run_generalization_benchmark
from evaluation.arithmetic import verify

model, tokenizer, ck = load_model_and_tokenizer(BEST_CK, device=DEVICE)
n_params = model.count_parameters()
step     = ck.get('global_step', '?')
cfg      = model.config

print('=== Model loaded ===')
print(f'  Parameters:    {n_params:,}')
print(f'  Training step: {step}')
print(f'  Architecture:  d{cfg.d_model}/L{cfg.n_layers}/H{cfg.n_heads}/ff{cfg.d_ff}')
print(f'  Vocab size:    {cfg.vocab_size}')
print(f'  Tokenizer:     {tokenizer.vocab_size} tokens')
print(f'  Device:        {DEVICE}')

# Generator — same settings used during training
generator = ArithmeticGenerator(seed=42, difficulty='medium')
print('\nReady for evaluation.')

---
## Step 4 — Test benchmark (500 unseen problems)

In [ ]:
print('Running test benchmark (500 problems, medium difficulty, greedy)...')
print('This takes ~2-3 minutes on T4 GPU.')
print()
t0 = time.time()

bench = run_benchmark(
    model=model, tokenizer=tokenizer, generator=generator,
    n_problems=500, split='test',
    max_new_tokens=64, temperature=0.0,
    use_reasoning=True, max_samples_to_save=50,
    device=DEVICE,
)

elapsed = time.time() - t0
print(bench.summary_str())
print(f'\nCompleted in {elapsed:.0f}s')

# Note if result changed significantly from the known baseline
known_baseline = 0.178
if abs(bench.accuracy - known_baseline) > 0.03:
    print(f'\nNOTE: Result differs from baseline ({known_baseline:.1%}) by more than 3%.')
    print('This may mean the test set was regenerated differently, or training parameters changed.')
else:
    print(f'\nResult consistent with known baseline ({known_baseline:.1%}).')

---
## Step 5 — Generalization benchmark (5 difficulty levels)

In [ ]:
# Generate exclusion set (all training problems) to ensure no overlap
print('Building exclusion set from training data (avoids evaluating on seen problems)...')
train_ex, val_ex, test_ex = generator.generate_all_splits(
    n_train=20000, n_val=2000, n_test=1000
)
all_known = set(e.problem for e in train_ex + val_ex + test_ex)
print(f'Exclusion set: {len(all_known):,} unique problems')
print()

print('Running generalization benchmark (200 problems per level, 5 levels)...')
print('This takes ~5-8 minutes on T4 GPU.')
print()
t0 = time.time()

gen_results = run_generalization_benchmark(
    model=model, tokenizer=tokenizer, generator=generator,
    n_per_level=200, max_new_tokens=64, temperature=0.0,
    device=DEVICE, use_reasoning=True,
    exclude_problems=all_known,
)

elapsed = time.time() - t0
LEVEL_DESC = {
    1: 'New values, same templates (medium)',
    2: 'Larger number ranges (hard difficulty)',
    3: 'Mixed families (medium)',
    4: 'Long reasoning chains (hard, multi_step+algebra)',
    5: 'All families, hard difficulty',
}

print(f'Completed in {elapsed:.0f}s')
print()
print(f'{"Lvl":<5} {"Correct":>8} {"Total":>7} {"Accuracy":>10}  Description')
print('-' * 72)

total_c, total_n = 0, 0
for level in sorted(gen_results.keys()):
    r = gen_results[level]
    total_c += r['correct']; total_n += r['total']
    print(f"{level:<5} {r['correct']:>8} {r['total']:>7} {r['accuracy']:>10.1%}  {LEVEL_DESC[level]}")

overall = total_c / total_n if total_n else 0.0
print('-' * 72)
print(f"{'ALL':<5} {total_c:>8} {total_n:>7} {overall:>10.1%}  Overall generalization")

# Save
gen_out = os.path.join(EXP_DIR, 'generalization_results.json')
os.makedirs(EXP_DIR, exist_ok=True)
with open(gen_out, 'w') as f:
    json.dump(gen_results, f, indent=2)
print(f'\nResults saved to: {gen_out}')

---
## Step 6 — Checkpoint progression (init → early → mid → final → best)

In [ ]:
# Probe problems — one per family, not cherry-picked
PROBE_PROBLEMS = [
    ('single_op',       'What is 37 + 58?',                                                            '95'),
    ('single_op',       'What is 6 * 9?',                                                              '54'),
    ('comparison',      'Which is greater: 44 or 51?',                                                 '51'),
    ('number_sequence', 'What is the next number in the sequence: 2, 4, 8, 16, 32, ...?',              '64'),
    ('multi_step',      'Calculate: 5 + 3 * 2',                                                        '11'),
    ('multi_step',      'Calculate: 12 + 8 - 3 * 2',                                                   '14'),
    ('percentage',      'What is 25% of 80?',                                                          '20'),
    ('percentage',      'What is 10% of 50?',                                                          '5'),
    ('ratio',           'Two quantities are in the ratio 3:2. If the total is 50, what is the first quantity?', '30'),
    ('algebra_linear',  'Solve for x: 2x + 4 = 10',                                                   '3'),
    ('algebra_linear',  'Solve for x: 5x + -3 = -23',                                                 '-4'),
    ('word_problem',    'Alice has 12 apples. She buys 7 more. How many apples does she have now?',    '19'),
]

# Find which progression checkpoints exist
LABELS   = ['init', 'early', 'mid', 'final', 'best']
available = []
print('Checking for progression checkpoints:')
for lb in LABELS:
    path = os.path.join(CK_DIR, f'{lb}.pt')
    exists = os.path.exists(path)
    print(f'  {lb}.pt  {"EXISTS" if exists else "missing"}')
    if exists:
        available.append(lb)

if len(available) <= 1:
    print()
    print('Only best.pt is available — checkpoint progression comparison will be skipped.')
    print('Progression checkpoints (init/early/mid/final) are only saved during training.')

In [ ]:
# Run probe problems through each available checkpoint
progression = {}

if len(available) > 1:
    print(f'Evaluating {len(PROBE_PROBLEMS)} probe problems across {len(available)} checkpoints...')
    for label in available:
        ck_path = os.path.join(CK_DIR, f'{label}.pt')
        m, tok, ck_i = load_model_and_tokenizer(ck_path, device=DEVICE)
        step_i = ck_i.get('global_step', '?')
        progression[label] = {'_step': step_i}
        for family, problem, expected in PROBE_PROBLEMS:
            result = generate_answer(problem=problem, model=m, tokenizer=tok,
                                     device=DEVICE, use_reasoning=True, max_new_tokens=64)
            try: exp_num = float(expected)
            except ValueError: exp_num = 0.0
            vr = verify(result['full_text'], expected, exp_num)
            progression[label][problem] = {
                'extracted': result['extracted_answer'],
                'correct':   vr['correct'],
            }
        del m
        if DEVICE == 'cuda': torch.cuda.empty_cache()
        n_ok = sum(1 for _, p, _ in PROBE_PROBLEMS if progression[label].get(p, {}).get('correct'))
        print(f'  [{label}] step={step_i}  correct={n_ok}/{len(PROBE_PROBLEMS)}')

    # Table
    print()
    hdr = f'{"Family":<18} {"Expected":<8}'
    for lb in available:
        hdr += f'  {lb[:6]}(s{str(progression[lb]["_step"])[:4]})'.ljust(13)
    print(hdr)
    print('-' * len(hdr))
    for family, problem, expected in PROBE_PROBLEMS:
        row = f'{family:<18} {expected:<8}'
        for lb in available:
            d = progression[lb].get(problem, {})
            mark = 'OK' if d.get('correct') else '--'
            pred = str(d.get('extracted', '?'))[:6]
            row += f'  {mark}({pred})'.ljust(13)
        print(row)
else:
    print('Skipped — only best.pt available. See Step 7 for best.pt outputs.')

---
## Step 7 — Full model outputs from best.pt

In [ ]:
print('=== Full outputs from best.pt ===')
print('Every probe problem is shown — correct and incorrect.')
print()

for family, problem, expected in PROBE_PROBLEMS:
    result = generate_answer(problem=problem, model=model, tokenizer=tokenizer,
                             device=DEVICE, use_reasoning=True, max_new_tokens=64)
    try: exp_num = float(expected)
    except ValueError: exp_num = 0.0
    vr = verify(result['full_text'], expected, exp_num)
    status = 'CORRECT' if vr['correct'] else 'WRONG  '
    print(f'[{status}] [{family}]  expected={expected}, got={vr["predicted"]}')
    print(result['full_text'])
    print()

---
## Step 8 — Failure analysis

In [ ]:
# Show representative failures from the full test benchmark
failures = [s for s in bench.samples if not s['correct']]
correct  = [s for s in bench.samples if     s['correct']]

print(f'From the first {len(bench.samples)} test samples:')
print(f'  Correct: {len(correct)}   Wrong: {len(failures)}')
print()
print('--- Failures by family (one per family) ---')
seen = set()
for s in failures:
    if s['family'] not in seen and len(seen) < 8:
        seen.add(s['family'])
        print(f"[WRONG] [{s['family']}]")
        print(f"  Problem:   {s['problem']}")
        print(f"  Expected:  {s['expected']}")
        print(f"  Predicted: {s['predicted']}")
        print(f"  Output:    {s['generated_text'][:200]}")
        print()

print('--- Correct answers (first 3) ---')
for s in correct[:3]:
    print(f"[OK] [{s['family']}] {s['problem']}")
    print(f"  Expected: {s['expected']}  |  Predicted: {s['predicted']}")
    print(f"  Output:   {s['generated_text'][:200]}")
    print()

---
## Step 9 — Summary report

In [ ]:
print('=' * 60)
print('EVALUATION SUMMARY')
print('=' * 60)
print()
print(f'Checkpoint:  {BEST_CK}')
print(f'Parameters:  {n_params:,}')
print(f'Step:        {step}')
print(f'Device:      {DEVICE}')
print()
print('Test benchmark (500 unseen medium problems):')
print(f'  Overall: {bench.correct}/{bench.total} = {bench.accuracy:.1%}')
for fam, acc in sorted(bench.family_accuracy.items(), key=lambda x: -x[1]):
    bar = '#' * int(acc * 25)
    print(f'  {fam:<25} {acc:>6.1%}  {bar}')

zero_fams = [f for f, a in bench.family_accuracy.items() if a == 0.0]
if zero_fams:
    print(f'\n  ZERO accuracy: {zero_fams}')
print()
print('Generalization benchmark (200/level, zero training overlap):')
for level in sorted(gen_results.keys()):
    r = gen_results[level]
    print(f'  Level {level}: {r["correct"]:>3}/{r["total"]} = {r["accuracy"]:.1%}  {LEVEL_DESC[level]}')
print(f'  Overall: {total_c}/{total_n} = {overall:.1%}')

if progression:
    print()
    print('Checkpoint progression (correct/12 probe problems):')
    for lb in available:
        n_c = sum(1 for _, p, _ in PROBE_PROBLEMS
                  if progression[lb].get(p, {}).get('correct'))
        print(f'  {lb} (step {progression[lb]["_step"]}): {n_c}/12')
print()
print('=' * 60)

---
## Step 10 — Interactive inference

Type your own problems below. The model will generate a reasoning chain and answer.

**Note:** This model achieves ~17.8% accuracy overall. It works best on
number sequences and simple comparisons. Multi-step arithmetic and
percentage calculations produce incorrect answers most of the time.
Outputs should be treated as experimental.

In [ ]:
# ── One-shot examples — runs automatically ────────────────────────────────────
example_problems = [
    ('What is the next number in the sequence: 3, 6, 12, 24, 48, ...?', '96'),
    ('Which is greater: 73 or 81?',                                      '81'),
    ('What is 37 + 58?',                                                 '95'),
    ('What is 25% of 80?',                                               '20'),
    ('Calculate: 4 + 3 * 2',                                             '10'),
]

print('=== One-shot examples (best.pt) ===')
print()
for problem, expected in example_problems:
    result = generate_answer(problem=problem, model=model, tokenizer=tokenizer,
                             device=DEVICE, use_reasoning=True, max_new_tokens=80)
    try: exp_num = float(expected)
    except ValueError: exp_num = 0.0
    vr = verify(result['full_text'], expected, exp_num)
    status = 'CORRECT' if vr['correct'] else 'WRONG  '
    print(f'[{status}] expected={expected}, got={vr["predicted"]}')
    print(result['full_text'])
    print()

In [ ]:
# ── Interactive mode ──────────────────────────────────────────────────────────
# Type a math problem and press Enter.
# Type 'exit' to stop.
#
# When running "Run all", this cell will wait for your input.
# If you want to skip it during Run all, add a # before interactive_inference.

from inference.generate import interactive_inference

interactive_inference(
    checkpoint_path=BEST_CK,
    device=DEVICE,
    max_new_tokens=128,
    temperature=0.0,
    show_extracted_answer=True,
    verify_answers=True,
)